In [48]:
import pandas as pd
import numpy as np

In [49]:
df = pd.read_csv("WineQT.csv")

In [50]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df.drop(columns=['quality', 'Id'])
y = np.where(df['quality'] > 5, 1, 0).reshape(-1,1)


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [51]:
class NeuralNetwork:

    def __init__(self, input_layer_size, hidden_layer_size, output_layer_size, epochs=1000, learning_rate = 0.01):
        self.input_layer_size = input_layer_size
        self.hidden_layer_size = hidden_layer_size
        self.output_layer_size = output_layer_size
        self.learning_rate = learning_rate
        self.epochs = epochs

        self.w1 = np.random.randn(input_layer_size, hidden_layer_size)
        self.b1 = np.zeros((1, hidden_layer_size))

        self.w2 = np.random.randn(hidden_layer_size, output_layer_size)
        self.b2 = np.zeros((1, output_layer_size))

    def sigmoid(self, x):
        return 1 / (1 + np.exp(-x))

    def sigmoid_derivative(self, a):
        return a * (1 - a)

    def forward(self, X):
        self.z1 = X @ self.w1 + self.b1
        self.a1 = self.sigmoid(self.z1)

        self.z2 = self.a1 @ self.w2 + self.b2
        self.a2 = self.sigmoid(self.z2)

        return self.a2

    def backward(self, X, y):

        delta2 = (self.a2 - y) * self.sigmoid_derivative(self.a2)

        self.dw2 = self.a1.T @ delta2
        self.db2 = np.sum(delta2, axis=0, keepdims=True)

        delta1 = (delta2 @ self.w2.T) * self.sigmoid_derivative(self.a1)

        self.dw1 = X.T @ delta1
        self.db1 = np.sum(delta1, axis=0, keepdims=True)

    def update_parameters(self):

        self.w1 -= self.learning_rate * self.dw1
        self.b1 -= self.learning_rate * self.db1

        self.w2 -= self.learning_rate * self.dw2
        self.b2 -= self.learning_rate * self.db2

    def mse(self, y_true, y_pred):
       return np.mean((y_true - y_pred) ** 2)

    def fit(self, X, y):

        for epoch in range(self.epochs):
           self.forward(X)

           loss = self.mse(y, self.a2)

           self.backward(X, y)

           self.update_parameters()

           if epoch % 100 == 0:

              print(f"Epoch {epoch}, Loss = {loss:.4f}")

    def predict(self, X):

        probabilities = self.forward(X)

        return (probabilities >= 0.5).astype(int)

In [52]:
network = NeuralNetwork(
    input_layer_size=11,
    hidden_layer_size=5,
    output_layer_size=1,
    learning_rate=0.01,
    epochs=1000
)
network.fit(X_train, y_train)
y_pred = network.predict(X_test)

Epoch 0, Loss = 0.2594
Epoch 100, Loss = 0.1640
Epoch 200, Loss = 0.1604
Epoch 300, Loss = 0.1586
Epoch 400, Loss = 0.1569
Epoch 500, Loss = 0.1551
Epoch 600, Loss = 0.1530
Epoch 700, Loss = 0.1511
Epoch 800, Loss = 0.1487
Epoch 900, Loss = 0.1460


In [53]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))

Accuracy: 0.7510917030567685
Precision: 0.7777777777777778
Recall: 0.7716535433070866
F1: 0.7747035573122529
